# Day 13: PDF Extraction (Start)

## Goal

Extract raw text from PDFs and get a first look at OCR for image-based documents.

By the end of this notebook you will have:

- extracted text from a PDF that already contains selectable text
- rendered a PDF page as an image
- tried OCR on an image-based example
- understood when to use direct extraction vs OCR

This is the starting point for document pipelines.

## Step 0: Sync the Project Once

The Day 13 Python packages are already included in `pyproject.toml`, so learners only need:

```bash
uv sync
```

For course maintainers, the Day 13 packages were added with:

```bash
uv add pypdf pymupdf pillow pytesseract
```

## Important Note About OCR

`pytesseract` is a Python wrapper, but OCR also requires the **Tesseract** system binary to be installed on the machine.

Typical install commands:

- macOS: `brew install tesseract`
- Ubuntu/Debian: `sudo apt-get install tesseract-ocr`
- Windows: install Tesseract and add it to PATH

The notebook will still work for PDF text extraction even if OCR is not installed.

In [1]:
import shutil
from pathlib import Path

import fitz
import pytesseract
from PIL import Image, ImageDraw
from pypdf import PdfReader

print("Imports loaded successfully.")

Imports loaded successfully.


In [3]:
sample_pdf = Path("../resources/AI_Hands_on_Training.pdf")
sample_pdf.exists(), sample_pdf

(True, PosixPath('../resources/AI_Hands_on_Training.pdf'))

## Step 1: Direct PDF Text Extraction

If a PDF already contains real text, we can extract it directly without OCR.

This is usually faster and cleaner than OCR.

In [4]:
reader = PdfReader(sample_pdf)
num_pages = len(reader.pages)
num_pages

4

In [5]:
def extract_text_from_pdf(pdf_path: Path, max_pages: int | None = None) -> str:
    reader = PdfReader(pdf_path)
    pages = reader.pages if max_pages is None else reader.pages[:max_pages]
    chunks = []
    for page in pages:
        chunks.append(page.extract_text() or "")
    return "\n\n".join(chunks)


raw_pdf_text = extract_text_from_pdf(sample_pdf, max_pages=3)
print(raw_pdf_text[:3000])

Week 1 – Setup + Core Understanding 
Day 1 – Basic Python API (Flask/FastAPI) 
• Simple CRUD 
• One endpoint: /query 
 Output: working API 
Day 2 – API + AI Integration 
• Call OpenAI/Ollama from API 
• Return AI response 
  Output: API + LLM working 
Day 3 – Docker + Folder Structure 
• Containerize API 
• Standard project structure 
  Output: runnable container 
Day 4 – Tokenization  
• Tokens, token IDs 
• Show cost impact 
 Output: prompt vs token comparison 
Day 5 – Embeddings + Vector DB (Chroma) 
• Create embeddings 
• Store + query 
 Output: similarity search working 
 

 
 
Week 2 – Advanced + Extensions 
Day 6 – Agent Skills 
• Tool calling 
• Multi-step flow 
 Output: simple agent example 
Day 7 – MCP Servers (Google Stitch) 
• Explain concept 
• Demo usage 
 Output: awareness + basic usage 
Day 8 – Dev Context (devctx walkthrough) 
• Show code-aware AI 
 Output: understanding, not deep build 
Day 9 – CopilotKit UI 
• Simple chat UI 
• Connect backend 
 Output: UI + backend 

## Step 2: Render a PDF Page as an Image

For scanned PDFs or image-heavy PDFs, we often need to convert pages into images before OCR.

In [ ]:
def render_pdf_page(pdf_path: Path, page_number: int = 0, zoom: float = 2.0) -> Image.Image:
    doc = fitz.open(pdf_path)
    page = doc.load_page(page_number)
    matrix = fitz.Matrix(zoom, zoom)
    pix = page.get_pixmap(matrix=matrix)
    image = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
    return image


page_image = render_pdf_page(sample_pdf, page_number=0)
page_image

## Step 3: Decide If OCR Is Available

The next cells detect whether the `tesseract` binary is installed.

In [ ]:
tesseract_path = shutil.which("tesseract")
tesseract_path

## Step 4: OCR the Rendered PDF Page

If Tesseract is installed, we can try OCR on the image version of the PDF page.

For digital PDFs, this is usually less clean than direct text extraction. The purpose here is to show the OCR path.

In [ ]:
if not tesseract_path:
    print("Tesseract is not installed or not on PATH. Skip OCR cells or install it first.")
else:
    ocr_text = pytesseract.image_to_string(page_image)
    print(ocr_text[:3000])

## Step 5: Create a Simple Invoice-Like Image for OCR

Image invoices often need OCR because they may come from scans, screenshots, or photos.

In [ ]:
invoice_image = Image.new("RGB", (1000, 500), color="white")
draw = ImageDraw.Draw(invoice_image)

invoice_lines = [
    "ACME SUPPLIES",
    "Invoice Number: INV-1007",
    "Invoice Date: 2026-03-21",
    "Vendor: ACME SUPPLIES PVT LTD",
    "Items: Printer Ink, Paper",
    "Total Amount: 12450 INR",
]

y = 40
for line in invoice_lines:
    draw.text((40, y), line, fill="black")
    y += 55

output_dir = Path("day13/generated")
output_dir.mkdir(parents=True, exist_ok=True)
invoice_image_path = output_dir / "sample_invoice.png"
invoice_image.save(invoice_image_path)

invoice_image

## Step 6: OCR the Invoice-Like Image

This is closer to what happens with scanned invoices.

In [ ]:
if not tesseract_path:
    print("Tesseract is not installed or not on PATH. Install it to run OCR.")
else:
    invoice_ocr_text = pytesseract.image_to_string(invoice_image)
    print(invoice_ocr_text)

## Step 7: Compare Direct Extraction vs OCR

Use direct extraction when:

- the PDF already contains real text
- you want cleaner output
- you want faster extraction

Use OCR when:

- the PDF is a scan
- the source is an image or screenshot
- direct extraction returns empty or unusable text

## Day 13 Recap

You now have the raw document extraction foundations:

- direct text extraction from a PDF
- conversion of a PDF page into an image
- OCR on an image-based example

The next step is to feed extracted document text into retrieval and question-answering pipelines.